# Footprint Reductions 

- update : 2026-09-09
- kernel : conda_py313_opsim53
- Ce notebook : download from : https://github.com/lsst-pst/survey_strategy/blob/main/fbs_5.3/v5.3_Update.ipynb
- summary.h5 : https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Opsim baseline runs : https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/baseline/
- Opsim other runs : https://s3df.slac.stanford.edu/data/rubin/sim-data/
- Table of simulations : https://usdf-maf.slac.stanford.edu/

In [ ]:
# import necessary and useful packages
import os
import numpy as np
import matplotlib.pyplot as plt
import colorcet
import cycler
import pandas as pd
import healpy as hp
import rubin_sim.maf as maf

In [ ]:
path_topsim = os.getenv("RUBIN_SIM_DATA_DIR")
path_summary = os.path.join(path_topsim, "maf/fbs5.3.6/summary.h5")
print(path_topsim, " -- ", path_summary)

Jump to comparison of metrics targeting: 
* [SCOC overview](#SCOC)
* [Science Requirements Document](#SRD)
* [DESC WFD coverage](#DESCWFD)
* [Transient and Variable Stars](#TVS)
* [Solar System Objects](#SSO)
* [Active Galactic Nuclei](#AGN)
* [Galaxies](#Galaxies)
* [Strong Lensing](#StrongLensing)
* [Stars](#Stars)
* [Milky Way and Local Volume](#MWLV)

## Get background information about summary statistics

Read the information about the metric sets in this version of the sims outputs. The metric_sets defines groups of summary statistics that have been identified as likely relevant for comparing simulations for particular science purposes. 

In [ ]:
metrics = maf.get_metric_subsets("metric_subsets.json")
msets = list(metrics.groupby("metric subset").first().index)

In [ ]:
summaries = maf.get_metric_summaries(summary_source=path_summary)
print(f"This summary h5 file contains information on {len(summaries.index)} simulations.")

mapper = {}
for b in "ugrizy":
    mapper.update(dict([(c, c.replace(f"band {b}", f"{b}")) for c in summaries.columns if f"band {b}" in c]))

checked = {}
for k, v in mapper.items():
    if k in summaries.columns and v in summaries.columns:
        checked[k] = v

for i, row in summaries.iterrows():
    for k, v in checked.items():
        if np.isnan(row[k]):
            row[k] = row[v]
        if np.isnan(row[v]):
            row[v] = row[k]

print(summaries.index)

In [ ]:
# check metric_subsets.json matches current metrics
for mset in msets:
    summaries[metrics.loc[mset].metric]

In [ ]:
m5_SRD_design = {"u": 23.90, "g": 25.00, "r": 24.70, "i": 24.00, "z": 23.30, "y": 22.10}
m5_SRD_minimum = {"u": 23.40, "g": 24.60, "r": 24.30, "i": 23.60, "z": 22.90, "y": 21.70}

# Weather variations on baseline strategy
weather_runs = [r for r in summaries.index if "weather" in r and "poor" not in r]
offsets = [int(w.split("dso")[-1].split("v5")[0]) for w in weather_runs]
idx = np.argsort(offsets)
weather_runs = [weather_runs[i] for i in idx]
print("weather runs", len(weather_runs))

# Start date variations
start_dates = [r for r in summaries.index if "start_date" in r]
offsets = [int(r.split("mjdp")[-1].split("_")[0]) for r in start_dates]
idx = np.argsort(offsets)
start_dates = [start_dates[i] for i in idx]
print("start_dates", len(start_dates))


fp_runs = [
    "shrink_fp_dust_0.050_v5.3.6_10yrs",
    "shrink_fp_dust_0.080_v5.3.6_10yrs",
    "shrink_fp_dust_0.120_v5.3.6_10yrs",
    "shrink_fp_dust_0.150_v5.3.6_10yrs",
    "shrink_fp_dust_0.199_v5.3.6_10yrs",
    "baseline_v5.3.6_10yrs",
    "shrink_fp_dust_0.250_v5.3.6_10yrs",
]
baseline = "baseline_v5.3.6_10yrs"

rdict = dict([(r, r.split("_v5.3.6_10yrs")[0]) for r in fp_runs])
runs = fp_runs

outdir = "tmp_fig"
try:
    os.mkdir(outdir)
except:
    pass

We can derive approximate uncertainties for some of our relevant metrics by looking at their variation across the different 'weather' instances of the baseline strategy.
This isn't necessarily appropriate for all metrics, but it does give a useful indications.

In [ ]:
dev = np.std(summaries.loc[weather_runs], axis=0)
hilo = np.abs(summaries.loc[weather_runs].max() - summaries.loc[weather_runs].min())

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

filter_colors = {
    "u": "#1600EA",
    "g": "#31DE1F",
    "r": "#B52626",
    "i": "#370201",
    "z": "#BA52FF",
    "y": "#61A2B3",
}

filter_rgb_map = {
    "u": (74 / 256, 125 / 256, 179 / 256),
    "g": (104 / 256, 173 / 256, 87 / 256),
    "r": (238 / 256, 134 / 256, 50 / 256),
    "i": (232 / 256, 135 / 256, 189 / 256),
    "z": (209 / 256, 53 / 256, 43 / 256),
    "y": (142 / 256, 82 / 256, 159 / 256),
}

filter_color_map = ListedColormap(list(filter_colors.values()))

## FP Comparison

In [ ]:
# opsdb = "baseline_v5.3.6_10yrs.db"
opsdb = os.path.join(path_topsim, "sim_baseline/baseline_v5.3.6_10yrs.db")

run_name = os.path.split(opsdb)[-1].replace(".db", "")

metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")
slicer = maf.HealpixSlicer(nside=64)
constraint = None
plot_dict = {"color_min": 0, "color_max": 1000, "extend": "max", "x_min": 0, "x_max": 1000}
plot_funcs = [maf.HealpixSkyMap(), maf.HealpixHistogram()]
bundle = maf.MetricBundle(
    metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
)

metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")
slicer = maf.HealpixSlicer(nside=64)
constraint = "night < 366"
plot_dict = {"color_min": 0, "color_max": 100, "extend": "max", "x_min": 0, "x_max": 100}
plot_funcs = [
    maf.HealpixSkyMap(),
]
bundle2 = maf.MetricBundle(
    metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
)

try_read = False
if try_read:
    bundle.read(bundle.file_root + ".npz")
    bundle.set_plot_dict(
        plot_dict={"color_min": 0, "color_max": 1000, "extend": "max", "x_min": 0, "x_max": 1000}
    )
    # bundle.set_plot_funcs(plot_funcs = [maf.HealpixSkyMap(),])
    bundle2.read(bundle2.file_root + ".npz")
    bundle2.set_plot_dict(
        plot_dict={"color_min": 0, "color_max": 100, "extend": "max", "x_min": 0, "x_max": 100, "bins": 100}
    )
    # bundle2.set_plot_funcs(plot_funcs = [maf.HealpixSkyMap(),])
else:
    g = maf.MetricBundleGroup({"nvisits": bundle, "year1": bundle2}, opsdb, verbose=True)
    g.run_all()

bundle.plot()

In [ ]:
bundle.set_plot_dict(
    plot_dict={
        "color_min": 0,
        "color_max": 1000,
        "extend": "max",
        "x_min": 0,
        "x_max": 1000,
        "coord": ["C", "G"],
    }
)
bundle.plot()

In [ ]:
bundle2.plot()

### Find shrink_fp_dust_XXX.db files:

https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/

In [ ]:
opsdb = "shrink_fp_dust_0.120_v5.3.6_10yrs.db"
opsdb = os.path.join(path_topsim, opsdb)

run_name = os.path.split(opsdb)[-1].replace(".db", "")

metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")
slicer = maf.HealpixSlicer(nside=64)
constraint = None
plot_dict = {"color_min": 0, "color_max": 1000, "extend": "max", "x_min": 0, "x_max": 1000}
plot_funcs = [maf.HealpixSkyMap(), maf.HealpixHistogram()]
fpbundle = maf.MetricBundle(
    metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
)

metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")
slicer = maf.HealpixSlicer(nside=64)
constraint = "night < 366"
plot_dict = {"color_min": 0, "color_max": 100, "extend": "max", "x_min": 0, "x_max": 100}
plot_funcs = [
    maf.HealpixSkyMap(),
]
fpbundle2 = maf.MetricBundle(
    metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
)

try_read = False
if try_read:
    fpbundle.read(bundle.file_root + ".npz")
    fpbundle.set_plot_dict(
        plot_dict={"color_min": 0, "color_max": 1000, "extend": "max", "x_min": 0, "x_max": 1000}
    )
    # fpbundle.set_plot_funcs(plot_funcs = [maf.HealpixSkyMap(),])
    fpbundle2.read(bundle2.file_root + ".npz")
    fpbundle2.set_plot_dict(
        plot_dict={"color_min": 0, "color_max": 100, "extend": "max", "x_min": 0, "x_max": 100, "bins": 100}
    )
    # fpbundle2.set_plot_funcs(plot_funcs = [maf.HealpixSkyMap(),])
else:
    g = maf.MetricBundleGroup({"nvisits": fpbundle, "year1": fpbundle2}, opsdb, verbose=True)
    g.run_all()

fpbundle.plot()

In [ ]:
fpbundle2.plot()

In [ ]:
# What is the difference in the *first year* coverage between these two runs?
# -- note that the dusty plane coverage changes (the WFD is not cutting off an area of dusty plane in the new run)
mval = bundle2.metric_values.filled(0) - fpbundle2.metric_values.filled(0)
hp.mollview(mval, min=-20, max=20)

In [ ]:
fpbundle.set_plot_dict(
    plot_dict={
        "color_min": 0,
        "color_max": 1000,
        "extend": "max",
        "x_min": 0,
        "x_max": 1000,
        "coord": ["C", "G"],
    }
)
fpbundle.plot()

In [ ]:
opsdb = "shrink_fp_dust_0.080_v5.3.6_10yrs.db"
opsdb = os.path.join(path_topsim, opsdb)


run_name = os.path.split(opsdb)[-1].replace(".db", "")

metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")
slicer = maf.HealpixSlicer(nside=64)
constraint = None
plot_dict = {"color_min": 0, "color_max": 1000, "extend": "max", "x_min": 0, "x_max": 1000}
plot_funcs = [maf.HealpixSkyMap(), maf.HealpixHistogram()]
ffbundle = maf.MetricBundle(
    metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
)

metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")
slicer = maf.HealpixSlicer(nside=64)
constraint = "night < 366"
plot_dict = {"color_min": 0, "color_max": 100, "extend": "max", "x_min": 0, "x_max": 100}
plot_funcs = [
    maf.HealpixSkyMap(),
]
ffbundle2 = maf.MetricBundle(
    metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
)

try_read = False
if try_read:
    ffbundle.read(bundle.file_root + ".npz")
    ffbundle.set_plot_dict(
        plot_dict={"color_min": 0, "color_max": 1000, "extend": "max", "x_min": 0, "x_max": 1000}
    )
    # ffbundle.set_plot_funcs(plot_funcs = [maf.HealpixSkyMap(),])
    ffbundle2.read(bundle2.file_root + ".npz")
    ffbundle2.set_plot_dict(
        plot_dict={"color_min": 0, "color_max": 100, "extend": "max", "x_min": 0, "x_max": 100, "bins": 100}
    )
    # ffbundle2.set_plot_funcs(plot_funcs = [maf.HealpixSkyMap(),])
else:
    g = maf.MetricBundleGroup({"nvisits": ffbundle, "year1": ffbundle2}, opsdb, verbose=True)
    g.run_all()

ffbundle.plot()

In [ ]:
ffbundle.set_plot_dict(
    plot_dict={
        "color_min": 0,
        "color_max": 1000,
        "extend": "max",
        "x_min": 0,
        "x_max": 1000,
        "coord": ["C", "G"],
    }
)
ffbundle.plot()

In [ ]:
runs

In [ ]:
fO = {}
for run_name in runs:
    db = run_name + ".db"
    if db != "baseline_v5.3.6_10yrs":
        db = os.path.join(path_topsim, db)
    else:
        db = os.path.join(path_topsim + "/sim_baseline", db)

    fO[run_name] = maf.fOBatch(run_name=run_name)
    g = maf.MetricBundleGroup(fO[run_name], db)
    g.run_all()

In [ ]:
# 18k sq deg version
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(8, 12))
colors = colorcet.glasbey_category10

# scale = area (1k sq deg) / npix
scale = hp.nside2pixarea(bundle.slicer.nside, degrees=True) / 1000.0
for count, name in enumerate(runs):
    bundle = fO[name][f"{name.replace('.', '_')}_fO_All_visits_HEAL"]
    nvis_sorted = np.sort(bundle.metric_values.filled(0))
    area = np.arange(30000, 0, -1)
    medianval = np.zeros(len(area))
    minval = np.zeros(len(area))
    for i, asky in enumerate(area):
        npix__asky = int(np.ceil(asky / hp.nside2pixarea(bundle.slicer.nside, degrees=True)))
        # Find the Asky's worth of healpixels with the largest # of visits.
        nvis__asky = nvis_sorted[-npix__asky:]
        # print(asky, npix__asky, nvis__asky[0], nvis__asky[-1], np.min(nvis__asky), np.max(nvis__asky), np.median(nvis__asky))
        medianval[i] = np.median(nvis__asky)
        minval[i] = np.min(nvis__asky)
    idx_15k = np.where(abs(area - 15000) < 0.001)[0]
    idx_18k = np.where(abs(area - 18000) < 0.001)[0]
    axes[0].plot(
        medianval,
        area / 1000,
        color=colors[count],
        linestyle="-",
        label=f"{name}: median Nv 18k sq deg {medianval[idx_18k].item():.0f}",
    )
    axes[0].plot(
        minval,
        area / 1000,
        linestyle=":",
        color=colors[count],
        label=f"{name}: minimum Nv 18k sq deg {minval[idx_18k].item():.0f}",
    )
    axes[1].plot(
        medianval,
        area / 1000,
        color=colors[count],
        linestyle="-",
        label=f"{name}: median Nv 15k sq deg {medianval[idx_15k].item():.0f}",
    )
    axes[1].plot(
        minval,
        area / 1000,
        linestyle=":",
        color=colors[count],
        label=f"{name}: minimum Nv 15k sq deg {minval[idx_15k].item():.0f}",
    )

for ax in axes:
    ax.legend(loc=(1.01, 0.1))
    ax.fill_between([750, 825], y1=[0], y2=[30], color="gray", alpha=0.2)
    ax.fill_betweenx([15, 18], x1=[0], x2=[1200], color="gray", alpha=0.2)
    ax.set_xlim(0, 1200)
    ax.set_ylim(0, 30)
    ax.set_xlabel("Number of visits", fontsize="x-large")
    ax.set_ylabel("Area (100s of square degrees) ", fontsize="x-large")
# plt.savefig("area_nvis.png", bbox_inches='tight')

In [ ]:
## PSTN-057 version
# runs = list(rdict.keys())
# runs = ['baseline_v3.0_10yrs', 'baseline_v3.6_10yrs', 'baseline_v4.0_10yrs',
#         'baseline_v5.0.1_10yrs', 'baseline_v5.1.2_10yrs',
#         'comp_survey_v5.3.0_10yrs', 'baseline_v5.3.1_10yrs', 'faster_templates_v5.3.0_10yrs', 'baseline_v5.3.0_11yrs']
# msub = metrics.loc['SCOC']
# baseline = 'baseline_v5.3.1_10yrs'


# fig, ax = maf.plot_run_metric_mesh(summaries.loc[runs, msub['metric']],
#                      baseline_run=baseline,
#                      metric_subset=msub,
#                         metric_label_map=msub['short_name'],
#                         run_label_map=rdict,
#                         color_range=0.4,
#                       )
# fig.set_figwidth(8)
# fig.set_figheight(8)
# fig.savefig(os.path.join(outdir, 'scoc_grid'+ '.png'), format='png', bbox_inches='tight')

In [ ]:
summaries.loc[
    runs, [c for c in summaries.columns if "Median Median" in c and "Healpix" in c and "all bands" in c]
].round(3)

In [ ]:
# Check in on the depths and number of visits in each bandpass
k = "WFD Depths"

msub = metrics.loc[k].query('metric.str.contains("Mean Median")')
display(msub)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]] - summaries.loc[baseline, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    cmap=filter_color_map,
    sep_plots=False,
)
ax.set_ylabel(r"$\Delta$ median visit m5 per pix (magnitude)", fontsize="large")
y1, y2 = plt.ylim()
fig.savefig(os.path.join(outdir, "visit_m5_per_pix" + ".png"), format="png")


msub = metrics.loc[k].query('metric.str.contains("Coadd")')
dislay(msub)
fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]] - summaries.loc[baseline, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    cmap=filter_color_map,
    sep_plots=False,
)
ax.set_ylabel(r"$\Delta$ median coadded depth (magnitude)", fontsize="large")
y1, y2 = plt.ylim()
fig.savefig(os.path.join(outdir, "coadd_m5_per_pix" + ".png"), format="png")

msub = metrics.loc[k].query('metric.str.contains("NVisits")')
display(msub)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]] - summaries.loc[baseline, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    cmap=filter_color_map,
    sep_plots=False,
)
ax.set_ylabel(r"$\Delta$ number of visits", fontsize="large")
y1, y2 = plt.ylim()
fig.savefig(os.path.join(outdir, "nvisit_per_pix" + ".png"), format="png")

summaries.loc[runs, metrics.loc[k]["metric"]].round(2)

In [ ]:
cols = [m for m in summaries if "Teff All sky all bands" in m and "UniSlicer" in m and "Normalized" not in m]
cols

d = summaries.loc[runs, cols] / summaries.loc[baseline, cols]
d = d.rename(columns={"Identity Total Teff All sky all bands UniSlicer": "Teff Total"})
d.round(2)

And finally, let's have a look at those depths compared to SRD design and minimum goals, to evaluate how these changes are redistributing depth (or, in a sense, time) among bandpasses.

In [ ]:
cols = [f"Identity Median fiveSigmaDepth WFD {f} band UniSlicer" for f in "ugrizy"]
newcols = [f"Median m5 WFD {f} band per image" for f in "ugrizy"]
visitdepth_cols = cols

mset = maf.create_metric_subset("VisitDepths", metrics=cols, short_name=newcols, invert=False, mag=True)

fig, ax = plt.subplots(figsize=(12, 5))

for f, col in zip(["u", "g", "r", "i", "z", "y"], cols):
    ax.plot(
        list(rdict.values()),
        summaries.loc[rdict.keys(), col] - m5_SRD_design[f],
        color=filter_rgb_map[f],
        marker=".",
        linestyle="-",
        label=f,
    )
    # ax.axhline(m5_SRD_design[f], color=filter_colors[f], linestyle=':', linewidth=3)

ax.axhline(0, color="k", linestyle="-")
ax.set_ylabel(f"Median m5 per WFD visit - SRD design", fontsize="large")
plt.legend(loc=(1.01, 0.35), fontsize="x-large")
ax.tick_params(axis="x", labelrotation=90, labelsize="large")
ax.grid()
fig.savefig(os.path.join(outdir, "m5_per_visit" + ".png"), format="png")

In [ ]:
## WFD coadd vs. exgal coadd depths

k = "WFD Depths"

msub = metrics.loc[k].query('metric.str.contains("Coadd")')
display(msub)
q1 = summaries.loc[runs, msub["metric"]]
names = {}
for m in msub["metric"]:
    names[m] = m.split(" band")[0][-1:]
q1.rename(columns=names, inplace=True)

mm = [f"Median Exgal_CoaddM5 WFD {f} band HealpixSubsetSlicer" for f in "ugrizy"]
q2 = summaries.loc[runs, mm]
names = {}
for m in metrics:
    names[m] = m.split(" band")[0][-1:]
q2.rename(columns=names, inplace=True)


summaries.loc[runs, mm].round(2)

## Evaluate metrics

Let's evaluate some metric results across these baseline surveys, starting with a very high level overview. 

In [ ]:
# How did the overall number of visits change for each simulation baseline?  Note some simulations include more shorter visits, and this will be reflected in these numbers.

cols = [m for m in summaries if "Nvisits" in m and "UniSlicer" in m and "All" in m]
cols

# cols = ['Sum young_stars HealpixSlicer']

fig, ax = maf.plot_run_metric_uncert(summaries.loc[runs, cols], dev[cols], run_label_map=rdict)
ax.set_ylabel(f"Number of Visits (Millions)", fontsize="large")
leg = ax.get_legend()
leg.remove()
# fig.savefig(os.path.join(outdir, 'total_nvisits'+ '.png'), format='png')

# summaries.loc[runs, cols] / summaries.loc['comp_survey_v5.3.0_10yrs', cols]

### SCOC subsets
<a id='SCOC'></a>
A high level overview in a mesh grid, to compare many metrics quickly but without details. 

In [ ]:
# What about the SCOC high-level overview?

msub = metrics.loc["SCOC"]
display(msub)

fig, ax = maf.plot_run_metric_mesh(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    color_range=0.4,
)
fig.set_figwidth(8)
fig.set_figheight(8)
fig.savefig(os.path.join(outdir, "scoc_grid" + ".png"), format="png")

In [ ]:
# And some classic 'radar' plots, but only use a subset of the baseline runs
rr = runs
rbaseline = baseline
print(rr)

mset = msub

statics = np.array([0, 1, 2, 3, 4, 13, 14, 15, 16, 17, 18])
variable = [i for i in np.arange(0, len(mset)) if i not in statics]

mset_stat = mset.iloc[statics]
mset_var = mset.iloc[variable]

normed = maf.normalize_metric_summaries(rbaseline, summaries.loc[rr, mset_stat["metric"]], mset)
normed.columns = normed.columns.map(mset["short_name"])
normed

fig, ax = maf.radar(normed, rgrids=[0.80, 0.9, 1.0, 1.1, 1.2, 1.3], figsize=(8, 8), bbox_to_anchor=[1.5, 0.1])
ax.tick_params(axis="x", labelsize="large")
ax.tick_params(axis="y", labelsize="medium")
fig.set_figwidth(8)
fig.set_figheight(7)


normed = maf.normalize_metric_summaries(rbaseline, summaries.loc[rr, mset_var["metric"]], mset)
normed.columns = normed.columns.map(mset["short_name"])
normed

fig, ax = maf.radar(normed, rgrids=[0.10, 0.9, 1.0, 1.1, 1.2, 1.8], figsize=(8, 8), bbox_to_anchor=[1.5, 0.1])
ax.tick_params(axis="x", labelsize="large")
ax.tick_params(axis="y", labelsize="medium")
fig.set_figwidth(8)
fig.set_figheight(7)
normed.loc[rr]

In [ ]:
# Plot all the metrics, resulting in a very big and messy plot. Look at overall trends, or for things to investigate further.
non_dd = [m for m in msets if ("DD" not in m)]
msub = metrics.loc[non_dd].reset_index(drop=True).drop("style", axis=1).drop("short_name", axis=1)
msub = msub.drop_duplicates().set_index("metric", drop=False)
display(msub)
fig, ax = maf.plot_run_metric_mesh(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    # metric_label_map=msub['short_name'],
    run_label_map=rdict,
    color_range=0.4,
)
# fig.set_figwidth(24)
fig.set_figheight(60)
fig.savefig(os.path.join(outdir, "all_grid" + ".png"), format="png")

Now we look at the metrics results for subsets of metrics, looking in more depth (which allows us to include error bars which come from the standard deviation of the metric results in the weather runs.

### SRD
<a id="SRD"></a>
Metrics coming from the Science Requirements Document, the [SRD](ls.st/srd). 

In [ ]:
msub = metrics.loc["SRD"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
)

### DESC WFD
<a id="DESC WFD"></a>
Metrics related to DESC WFD goals. These include both static science and transient science.

In [ ]:
msub = metrics.loc["DESC WFD"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
)

In [ ]:
# Compare 3x2pt FOM metrics at individual years, between different simulations.
m1 = []
for yr in range(1, 11):
    m1 += [f"3x2ptFoM Exgalm5WithCuts i band non-DD year {yr} HealpixSlicer"]

mset = maf.create_metric_subset(
    "3x2pt",
    metrics=m1,
    short_name=[m.replace(" HealpixSlicer", "") for m in m1],
)
msub = mset.loc["3x2pt"]

display(msub)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    cmap=colorcet.glasbey_bw,
    metric_subset=msub,
    markers=[
        "o",
        "x",
        "^",
        "s",
        "<",
        ">",
        ".",
    ],
    sep_plots=False,
)
ax.set_ylabel("3x2pt FoM", fontsize="x-large")
fig.set_figwidth(8)
fig.set_figheight(5)

### Transient and Variable Stars
<a id="TVS"></a>
Metrics related to Transient and Variable Stars collaboration goals. 

In [ ]:
msub = metrics.loc["TVS short"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
)

In [ ]:
# Time Gaps metric

fig, ax = plt.subplots()
mset = metrics.loc["TVS Gaps"][0:8]

run_names = [rdict[r] for r in runs]


markers = ["o", "s", "*", "<", "^", "v", "*", ">", ".", "X"]
j = 0
for i, mm in mset.iterrows():
    m = mm["metric"]
    ff = mm["short_name"].split("in ")[-1].split(" at")[0]
    # marker= markers[j]
    marker = "o"
    if "3" in m:
        marker = "^"
        linestyle = "-"
        alpha = 0.8
        label = f"{ff} 3 hr"
    if "7" in m:
        marker = "o"
        linestyle = ":"
        alpha = 0.6
        label = f"{ff} 7 hr"
    j += 1
    ax.errorbar(
        run_names,
        summaries.loc[runs, m],
        yerr=dev[m],
        color=filter_rgb_map[ff],
        alpha=alpha,
        marker=marker,
        linestyle=linestyle,
        label=label,
    )
plt.legend(loc=(1.01, 0.65), fontsize="x-large", ncols=2)
ax.set_ylabel("Mean 'Gaps'", fontsize="x-large")
ax.tick_params(axis="x", labelrotation=90, labelsize="large")
fig.savefig(os.path.join(outdir, "tgaps" + ".png"), format="png")

In [ ]:
msub = metrics.loc["TVS Gaps"][8:12]
display(msub)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    cmap=colorcet.glasbey_dark,
    metric_subset=msub,
    sep_plots=False,
)
plt.legend(loc=(1.01, 0.3), fontsize="x-large", ncols=2)
ax.set_ylabel("Mean 'Gaps'", fontsize="x-large")
ax.tick_params(axis="x", labelrotation=90, labelsize="large")

In [ ]:
msub = metrics.loc["TVS KNe short"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
)

In [ ]:
msub = metrics.loc["TVS microlensing short"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=True,
)

msub = metrics.loc["TVS microlensing all"]
fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

In [ ]:
msub = metrics.loc["TVS anomalies"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)
ax.legend(loc=(1.01, 0.3))

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
)

### Solar System Objects
<a id="SSO"></a>
Metrics related to Solar System Science Collaboration goals. 

In [ ]:
class TTotalNumberSSO(maf.BaseMoMetric):
    """Calculate the total number of objects of a given population
    expected at a given H value or larger.

    Operations on differential completeness values
    (or equivalent; fractions of the population is ok if
    still a differential metric result, not cumulative).

    Parameters
    ----------
    h_mark : `float`, optional
        The H value at which to calculate the expected total number of objects.
    dndh_func : function, optional
        The dN/dH distribution used calculate the expected population size.

    Returns
    -------
    nObj : `float`
        The predicted number of objects in the population.
    """

    def __init__(self, h_mark=22, dndh_func=maf.neo_dndh_granvik, dndh_args=None, **kwargs):
        self.h_mark = h_mark
        self.dndh_func = dndh_func
        self.dndh_args = dndh_args
        metric_name = "Nobj <= %.1f" % (h_mark)
        super().__init__(metric_name=metric_name, **kwargs)

    def run(self, metric_vals, h_vals):
        totals = maf.sum_over_h(metric_vals, h_vals, self.dndh_func, **self.dndh_args)
        n_obj = np.interp(self.h_mark, h_vals, totals)
        return n_obj

In [ ]:
# Plot completeness over time
# mm = [c for c in summaries if 'Vatira' in c and 'CumulativeCompleteness' in c and 'DiscoveryTime' in c
#            and 'quad' in c and ' detection loss' in c and "H <=20.0" in c and "SNR" not in c]
# times = [float(m.split("@ ")[1].split(" DiscoveryTime")[0]) for m in mm]
# times = np.array(times)
# mvals = np.array([summaries.loc[baseline, m] for m in mm])
# good = np.where(~np.isnan(mvals))

# plt.plot(times[good], mvals[good])

# mm = [c for c in summaries if 'Vatira' in c and 'CumulativeCompleteness' in c and 'DiscoveryTime' in c
#            and '3 pairs in 15 nights' in c and ' detection loss' in c and "H <=20.0" in c and "SNR" not in c]
# times = [float(m.split("@ ")[1].split(" DiscoveryTime")[0]) for m in mm]
# times = np.array(times)
# mvals = np.array([summaries.loc[baseline, m] for m in mm])
# good = np.where(~np.isnan(mvals))

# plt.plot(times[good], mvals[good])

In [ ]:
# # Calculate total #'s of objects
# mm = [c for c in summaries if 'NEO' in c and 'DifferentialCompleteness' in c and 'DiscoveryNChances' in c
#            and '3 pairs in 15 nights ' in c and ' detection loss' in c and 'SNR' not in c]
# hvals = [float(m.split("= ")[1].split(" DiscoveryNChances")[0]) for m in mm]
# hrange = np.sort(np.array(hvals))

# # NTr5 = 794 × 10^0.44(H−12)
# # NTNO = 71,400 × 10^0.63(H−9.1),
# # NNEA = 960 × 10^0.35(H−18)
# kwargs = {'no': 794, 'ho': 12, 'h_index': 0.44}
# kwargs = {'no': 960, 'ho': 18, 'h_index': 0.35}
# kwargs = {'no': 71400, 'ho': 9.1, 'h_index': 0.63}
# total_num = TTotalNumberSSO(h_mark=26, dndh_func=maf.neo_dndh_granvik, dndh_args=kwargs)
# #total_num = TTotalNumberSSO(h_mark=22, dndh_func=maf.power_law_dndh, dndh_args=kwargs)
# for r in runs:
#     _ = plt.plot(hrange, summaries.loc[r, mm].values)
#     nobj = total_num.run(summaries.loc[r, mm].values, hrange)
#     print(r, nobj)
# #50 percent H value
# #np.interp(0.5, 1-summaries.loc[r, mm].values, hrange)

In [ ]:
msub = metrics.loc["SSO discovery"]
display(msub)


fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

msub = metrics.loc["SSO discovery"][::2]

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=False,
)
fig.savefig(os.path.join(outdir, "completeness_bright" + ".png"), format="png")

msub = metrics.loc["SSO discovery"][1::2]
fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=False,
)
fig.savefig(os.path.join(outdir, "completeness_faint" + ".png"), format="png")


msub = metrics.loc["SSO lightcurve inversion"]
fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)


msub = metrics.loc["SSO lightcurve inversion"][1::2]
fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=False,
)
fig.savefig(os.path.join(outdir, "lightcurves_faint" + ".png"), format="png")


msub = metrics.loc["SSO fraction 6 bands"]
fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

msub = metrics.loc["SSO fraction 6 bands"][::2]
fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=False,
)
fig.savefig(os.path.join(outdir, "colors6_bright" + ".png"), format="png")

msub = metrics.loc["SSO fraction 6 bands"][1::2]
fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=False,
)
fig.savefig(os.path.join(outdir, "colors6_faint" + ".png"), format="png")

msub = metrics.loc["SSO fraction 5 bands"]
fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

msub = metrics.loc["SSO fraction 4 bands"]
fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

msub = metrics.loc["SSO fraction 3 bands"]
fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

### AGN
<a id="AGN"></a>
Metrics contributed by the Active Galactic Nuclei science collaboration.

In [ ]:
msub = metrics.loc["AGN short"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)
ax.legend(loc=(1.01, 0.3))

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=True,
)

In [ ]:
msub = metrics.loc["AGN SF"]
display(msub)

u = msub.iloc[0:1]
g = msub.iloc[1:2]
r = msub.iloc[2:3]
agn_sf = pd.DataFrame(
    summaries.loc[runs, u["metric"]].values * 2
    + summaries.loc[runs, g["metric"]].values * 1
    + summaries.loc[runs, r["metric"]].values * 1,
    columns=["AGN SF Error (2*u + g + r)"],
    index=runs,
)

fig, ax = maf.plot_run_metric(
    agn_sf, shade_fraction=None, linestyles="-", vertical_quantity="value", horizontal_quantity="run"
)
ylim = ax.set_ylim()
ax.set_ylim(ylim[1], ylim[0])
_ = plt.title("AGN SF Error combined (2*u+g+r)")

In [ ]:
msub = metrics.loc["AGN timelag"]
display(msub)


msub = msub.query('metric.str.contains("100")')
u = msub.iloc[0:1]
g = msub.iloc[1:2]
r = msub.iloc[2:3]
agn_sf = pd.DataFrame(
    summaries.loc[runs, u["metric"]].values * 2
    + summaries.loc[runs, g["metric"]].values * 1
    + summaries.loc[runs, r["metric"]].values * 1,
    columns=["AGN TimeLag 100 days (2*u + g + r)"],
    index=runs,
)

fig, ax = maf.plot_run_metric(
    agn_sf, shade_fraction=None, linestyles="-", vertical_quantity="value", horizontal_quantity="run"
)
ylim = ax.set_ylim()
_ = plt.title("AGN Timelag 100 days (2*u+g+r)")

### Galaxies
<a id="Galaxies"></a>
Metrics evaluating the total number of galaxies expected in the survey.

In [ ]:
msub = metrics.loc["galaxies"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=False,
)

### Strong Lensing
<a id="StrongLensing"></a>
Metrics contributed by the Strong Lensing Science Collaboration. 

In [ ]:
msub = metrics.loc["SL TDC"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=True,
)

In [ ]:
msub = metrics.loc["SL IQ"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=True,
)

### Stars
<a id="Stars"></a>
Metrics considering total number of stars available in the survey. 

In [ ]:
msub = metrics.loc["Stars"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

msub = metrics.loc["Stars"][0:6]
fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    cmap=filter_color_map,
    metric_subset=msub,
    sep_plots=False,
)

msub = metrics.loc["Stars"][6:]
fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=True,
)

In [ ]:
for f in "ugrizy":
    mm = [m for m in summaries if "Total N Stars" in m and f" {f} " in m]
    fig, ax = plt.subplots()
    for m, marker, color in zip(mm, ["o", "x"], ["black", "gray"]):
        ax.errorbar(
            list(rdict.values()),
            summaries.loc[runs, m],
            yerr=dev[m],
            color=color,
            marker=marker,
            linestyle="-",
            label=m.rstrip(" HealpixSlicer"),
        )
    ax.legend()
    ax.set_ylabel(f"Total N Stars {f} band")
    ax.tick_params(axis="x", labelrotation=90, labelsize="large")
    ax.grid(True, alpha=0.5)

### Milky Way & Local Volume 
<a id="MWLV"></a>
Metrics for the local volume and milky way. These metrics tend to have a focus on galactic plane coverage, south celestial pole or SMC & LMC coverage. 


In [ ]:
msub = metrics.loc["Local Volume"]
display(msub)

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, msub["metric"]],
    baseline_run=baseline,
    metric_subset=msub,
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    horizontal_quantity="run",
    vertical_quantity="value",
)

fig, ax = maf.plot_run_metric_uncert(
    summaries.loc[runs, msub["metric"]],
    dev[msub["metric"]],
    metric_label_map=msub["short_name"],
    run_label_map=rdict,
    metric_subset=msub,
    sep_plots=True,
)

In [ ]:
# First let's check the total number of visits in the DDFS,
# as this has a significant impact on the depths and everything else.
count_metrics = [
    "Identity Number of Exposures UniSlicer",
    "Identity Counts_DD:COSMOS UniSlicer",
    "Identity Counts_DD:ECDFS UniSlicer",
    #'Identity Counts_DD:EDFS, a UniSlicer',
    #'Identity Counts_DD:EDFS, b UniSlicer',
    "Identity Counts_DD:EDFS_a UniSlicer",
    "Identity Counts_DD:EDFS_b UniSlicer",
    "Identity Counts_DD:ELAISS1 UniSlicer",
    #'Identity Counts_DD:XMM-LSS UniSlicer',
    "Identity Counts_DD:XMM_LSS UniSlicer",
]

fig, ax = maf.plot_run_metric(
    summaries.loc[runs, count_metrics],
    baseline_run=baseline,
    linestyles="-",
    markers=[""],
    horizontal_quantity="run",
    vertical_quantity="value",
    shade_fraction=0.05,
)
# ax.set_ylim(0.3, 1.8)

display(summaries.loc[runs, count_metrics])

print("Percent of visits in DDF")
display(
    (100 * summaries.loc[runs, count_metrics[1:]].sum(axis=1) / summaries.loc[runs, count_metrics[0]]).round(
        2
    )
)